# exp37 — planted-backdoor detection on Pythia-160M (transformer-scale evidence)

Traceprop-only (no LogIX). Plants a trigger into K training examples (label flipped to a
target class) + distractors (trigger, correct label), trains with inline capture, and checks
whether attribution ranks the poisoned examples via **auc_within_target** (AUC over target-label
candidates only — isolates the trigger from the label-sign artifact). Clones `main`.

## Cell 1 — GPU check + mount Drive

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE")
assert any(g in gpu for g in ("A100","L4")), f"Need A100/L4, got {gpu!r}"
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Clone main + install deps

In [ ]:
%cd /content
!rm -rf Traceprop && git clone -q https://github.com/AmitoVrito/Traceprop.git
%cd /content/Traceprop
!git checkout -q main
!git log --oneline -1
import subprocess as _sp
_fix = _sp.run(["grep","-q","_tok.pad_token_id","experiments/exp27_lds_quality.py"]).returncode == 0
print("PAD-TOKEN FIX PRESENT:", _fix)
assert _fix, "STALE CLONE -- re-open the notebook fresh from GitHub"
!pip -q install "datasets>=2.14" "peft>=0.7" scikit-learn
!pip -q uninstall -y torchao   # Colab's torchao 0.10 breaks peft 0.20 LoRA dispatch
!pip -q install -e . --no-deps
import torch, transformers, peft, sklearn, datasets
print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | sklearn {sklearn.__version__}")
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Cell 3a — GAP CHECK (seed 0 only): did the backdoor get learned (gap>=0.3)?

In [ ]:
import subprocess, sys, os, json
os.chdir('/content/Traceprop/experiments')
cmd = [sys.executable, "exp37_planted_detection.py", "--backend","hf","--model","EleutherAI/pythia-160m","--data","sst2",
       "--device","cuda","--gpu_check","","--track","0",
       "--n_train","2000","--seq","64","--epochs","15","--lr","2e-4","--batch","16",
       "--plant_frac","0.15","--distractor_frac","0.05","--trigger_len","8",
       "--backdoor_target_label","1",
       "--n_seeds", "1", "--out", "results/exp37_gapcheck.json", "--force"]
p = subprocess.run(cmd, capture_output=True, text=True)
os.makedirs("/content/drive/MyDrive/traceprop_results/", exist_ok=True)
open("/content/drive/MyDrive/traceprop_results/exp37_gapcheck_stdout.txt","w").write(p.stdout)
if p.returncode != 0:
    print("EXIT", p.returncode, "STDERR tail:")
    print("\n".join(p.stderr.splitlines()[-40:]))
    raise SystemExit("exp37 gap check failed")
for ln in p.stdout.splitlines():
    if any(k in ln for k in ("backdoor_success", "learned", "backdoor-", "WARNING")):
        print(ln)
d = json.load(open("results/exp37_gapcheck.json"))
print("\nGAP CHECK -> per_seed_backdoor_learned:", d.get("per_seed_backdoor_learned"))
print("If [True] (gap>=0.3): run Cell 3b (full 5 seeds).")
print("If [False]: raise --plant_frac (0.1->0.15) and/or --epochs (10->15) in BOTH cells, re-run 3a.")

## Cell 3b — FULL run (5 seeds) — only after 3a shows learned=[True]

In [ ]:
import subprocess, sys, os, shutil
os.chdir('/content/Traceprop/experiments')
cmd = [sys.executable, "exp37_planted_detection.py", "--backend","hf","--model","EleutherAI/pythia-160m","--data","sst2",
       "--device","cuda","--gpu_check","","--track","0",
       "--n_train","2000","--seq","64","--epochs","15","--lr","2e-4","--batch","16",
       "--plant_frac","0.15","--distractor_frac","0.05","--trigger_len","8",
       "--backdoor_target_label","1",
       "--n_seeds", "5", "--out", "results/exp37_pythia160m_sst2.json", "--force"]
p = subprocess.run(cmd, capture_output=True, text=True)
os.makedirs("/content/drive/MyDrive/traceprop_results/", exist_ok=True)
open("/content/drive/MyDrive/traceprop_results/exp37_stdout.txt","w").write(p.stdout)
open("/content/drive/MyDrive/traceprop_results/exp37_stderr.txt","w").write(p.stderr)
if p.returncode != 0:
    print("EXIT", p.returncode, "STDERR tail:")
    print("\n".join(p.stderr.splitlines()[-45:]))
    raise SystemExit("exp37 full run failed")
for ln in p.stdout.splitlines():
    if any(k in ln for k in ("backdoor_success", "backdoor-", "learned", "NO seed", "WARNING", "n_valid_seeds")):
        print(ln)
for f in ("results/exp37_pythia160m_sst2.json", "results/exp37_pythia160m_sst2_raw.npz"):
    if os.path.exists(f): shutil.copy(f, "/content/drive/MyDrive/traceprop_results/")
print("saved to Drive")